In [1]:
import subprocess
import librosa
import numpy as np
import json
from pytube import YouTube
import os
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw

def dynamic_time_warping_approx(S1, S2):
    # Transpose the spectrogram matrices to align dimensions appropriately
    # fastdtw expects the format [time_steps x features]
    S1 = S1.T  # Transpose to switch dimensions to [time x features]
    S2 = S2.T

    # Perform FastDTW
    distance, path = fastdtw(S1, S2, dist=euclidean)
    return path

def download_audio(youtube_url, output_path):
    yt = YouTube(youtube_url)
    audio_stream = yt.streams.filter(only_audio=True).first()
    downloaded_file = audio_stream.download(filename=output_path)
    return downloaded_file

def convert_to_ogg(input_path, output_path):
    # Convert to OGG using ffmpeg
    subprocess.run(['ffmpeg', '-i', input_path, '-c:a', 'libvorbis', output_path], check=True)
    return output_path

def load_and_preprocess_audio(audio_path, target_sr=11025):
    # Load audio file at a reduced sample rate
    y, sr = librosa.load(audio_path, sr=target_sr)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    log_S = librosa.power_to_db(S, ref=np.max)
    return log_S, sr


def adjust_timestamps(wp, timestamps, sr, offset):
    mapping = {row[0]: row[1] for row in wp}
    adjusted_timestamps = []
    for entry in timestamps:
        original_frame = int((entry['t'] + offset) * sr / 1024)
        if original_frame in mapping:
            adjusted_time = mapping[original_frame] * 1024 / sr - offset
            adjusted_timestamps.append({"t": adjusted_time, "mix": entry['mix']})
    return adjusted_timestamps


# URLs for YouTube videos
youtube_url1 = 'https://www.youtube.com/watch?v=wxB5vkZy7nM'
youtube_url2 = 'https://www.youtube.com/watch?v=oUWKdARd-24'

# Download audio files
audio_path1 = download_audio(youtube_url1, 'youtube_audio1.mp3')
audio_path2 = download_audio(youtube_url2, 'youtube_audio2.mp3')

# Convert to OGG
ogg_path1 = convert_to_ogg(audio_path1, 'youtube_audio1.ogg')
ogg_path2 = convert_to_ogg(audio_path2, 'youtube_audio2.ogg')

# Load and preprocess both audio recordings in OGG format
S1, sr1 = load_and_preprocess_audio(ogg_path1)
S2, sr2 = load_and_preprocess_audio(ogg_path2)

# Dynamic Time Warping
warping_path = dynamic_time_warping_approx(S1, S2)


# Sample JSON timestamps for the first recording
timestamps_json = '''
[{"t":0,"mix":0},{"t":1.116,"mix":1},{"t":3.155,"mix":2},{"t":5.057,"mix":3},{"t":6.961,
"mix":4},{"t":8.784,"mix":5},{"t":10.749,"mix":6},{"t":12.513,"mix":7},{"t":14.477,"mix":8},
{"t":16.232,"mix":9},{"t":17.987,"mix":10},{"t":19.567,"mix":11},{"t":21.326,"mix":12},{"t":23.078,
"mix":13},{"t":24.857,"mix":14},{"t":26.728,"mix":15},{"t":28.736,"mix":16},{"t":30.528,"mix":17},
{"t":32.729,"mix":18},{"t":34.78,"mix":19},{"t":36.7,"mix":20},{"t":38.417,"mix":21},{"t":39.919,
"mix":22},{"t":41.555,"mix":23},{"t":43.15,"mix":24},{"t":44.789,"mix":25},{"t":46.397,"mix":26},
{"t":48.096,"mix":27},{"t":49.808,"mix":28},{"t":51.517,"mix":29},{"t":53.323,"mix":30},{"t":55.016,
"mix":31},{"t":56.722,"mix":32},{"t":58.361,"mix":33},{"t":60.112,"mix":34},{"t":61.767,"mix":35},
{"t":63.592,"mix":36},{"t":65.194,"mix":37},{"t":66.9,"mix":38},{"t":68.501,"mix":39},{"t":70.207,
"mix":40},{"t":71.783,"mix":41},{"t":73.329,"mix":42},{"t":75.019,"mix":43},{"t":76.656,"mix":44},
{"t":78.811,"mix":45},{"t":80.669,"mix":46},{"t":82.367,"mix":47},{"t":84.009,"mix":48},{"t":85.584,
"mix":49},{"t":87.364,"mix":50},{"t":89.109,"mix":51},{"t":90.655,"mix":52},{"t":92.408,"mix":53},
{"t":93.853,"mix":54},{"t":95.495,"mix":55},{"t":97.168,"mix":56},{"t":98.75,"mix":57},{"t":100.467,
"mix":58},{"t":102.084,"mix":59},{"t":103.703,"mix":60},{"t":105.34,"mix":61},{"t":106.947,
"mix":62},{"t":108.57,"mix":63},{"t":110.248,"mix":64},{"t":111.837,"mix":65},{"t":113.486,
"mix":66},{"t":115.029,"mix":67},{"t":116.625,"mix":68},{"t":118.16,"mix":69},{"t":119.667,
"mix":70},{"t":121.218,"mix":71},{"t":122.814,"mix":72},{"t":124.461,"mix":73},{"t":126.145,
"mix":74},{"t":127.778,"mix":75},{"t":129.31,"mix":76},{"t":131.059,"mix":77},{"t":132.719,
"mix":78},{"t":134.278,"mix":79},{"t":135.884,"mix":80},{"t":137.448,"mix":81},{"t":138.94,
"mix":82},{"t":140.384,"mix":83},{"t":141.932,"mix":84},{"t":143.342,"mix":85},{"t":144.831,
"mix":86},{"t":146.292,"mix":87},{"t":147.836,"mix":88},{"t":149.326,"mix":89},{"t":150.893,
"mix":90},{"t":152.681,"mix":91},{"t":154.634,"mix":92},{"t":156.541,"mix":93},{"t":158.599,
"mix":94},{"t":161.043,"mix":95},{"t":162.664,"mix":96},{"t":164.43,"mix":97},{"t":166.079,
"mix":98},{"t":167.79,"mix":99},{"t":169.557,"mix":100},{"t":171.204,"mix":101},{"t":172.893,
"mix":102},{"t":174.546,"mix":103},{"t":176.262,"mix":104},{"t":178.02,"mix":105},{"t":179.766,
"mix":106},{"t":181.567,"mix":107},{"t":183.548,"mix":108},{"t":185.308,"mix":109},{"t":187.024,
"mix":110},{"t":189.096,"mix":111},{"t":190.779,"mix":112},{"t":192.714,"mix":113},{"t":194.51,
"mix":114},{"t":196.513,"mix":115},{"t":198.215,"mix":116},{"t":200.023,"mix":117},{"t":201.748,
"mix":118},{"t":203.469,"mix":119},{"t":205.233,"mix":120},{"t":207.118,"mix":121},{"t":208.834,
"mix":122},{"t":210.449,"mix":123},{"t":212.225,"mix":124},{"t":213.841,"mix":125},{"t":215.438,
"mix":126},{"t":217.225,"mix":127},{"t":218.939,"mix":128},{"t":220.585,"mix":129},{"t":222.274,
"mix":130},{"t":223.873,"mix":131},{"t":225.489,"mix":132},{"t":227.125,"mix":133},{"t":228.744,
"mix":134},{"t":230.376,"mix":135},{"t":232.129,"mix":136},{"t":234.28,"mix":137},{"t":236.132,
"mix":138},{"t":237.907,"mix":139},{"t":239.767,"mix":140},{"t":241.573,"mix":141},{"t":243.457,
"mix":142},{"t":245.205,"mix":143},{"t":246.92,"mix":144},{"t":249.887,"mix":145},{"t":251.788,
"mix":146},{"t":253.505,"mix":147},{"t":255.219,"mix":148},{"t":257.052,"mix":149},{"t":258.868,
"mix":150},{"t":260.766,"mix":151},{"t":262.67,"mix":152},{"t":264.503,"mix":153},{"t":266.106,
"mix":154},{"t":267.97,"mix":155},{"t":269.773,"mix":156},{"t":271.6,"mix":157},{"t":273.388,
"mix":158},{"t":275.038,"mix":159},{"t":276.543,"mix":160},{"t":278.097,"mix":161},{"t":279.83,
"mix":162},{"t":281.45,"mix":163},{"t":283.057,"mix":164},{"t":284.714,"mix":165},{"t":286.426,
"mix":166},{"t":288.109,"mix":167},{"t":289.845,"mix":168},{"t":291.557,"mix":169},{"t":293.244,
"mix":170},{"t":294.79,"mix":171},{"t":296.406,"mix":172},{"t":298.037,"mix":173},{"t":299.632,
"mix":174},{"t":301.193,"mix":175},{"t":302.729,"mix":176},{"t":304.381,"mix":177},{"t":306.013,
"mix":178},{"t":307.51,"mix":179},{"t":309.134,"mix":180},{"t":310.767,"mix":181},{"t":312.363,
"mix":182},{"t":314.084,"mix":183},{"t":316.035,"mix":184},{"t":317.885,"mix":185},{"t":319.65,
"mix":186},{"t":321.435,"mix":187},{"t":323.272,"mix":188},{"t":325.497,"mix":189},{"t":327.598,
"mix":190},{"t":329.4,"mix":191},{"t":331.361,"mix":192},{"t":333.091,"mix":193},{"t":334.875,
"mix":194},{"t":336.588,"mix":195},{"t":338.42,"mix":196},{"t":340.32,"mix":197},{"t":341.99,
"mix":198},{"t":343.77,"mix":199},{"t":345.649,"mix":200},{"t":347.563,"mix":201},{"t":349.656,
"mix":202},{"t":351.586,"mix":203},{"t":353.289,"mix":204},{"t":355.177,"mix":205},{"t":357.19,
"mix":206},{"t":359.482,"mix":207},{"t":361.18,"mix":208},{"t":362.69,"mix":209},{"t":364.276,
"mix":210},{"t":365.883,"mix":211},{"t":367.437,"mix":212},{"t":368.934,"mix":213},{"t":370.53,
"mix":214},{"t":372.219,"mix":215},{"t":373.711,"mix":216},{"t":375.308,"mix":217},{"t":377.018,
"mix":218},{"t":378.632,"mix":219},{"t":380.693,"mix":220},{"t":382.688,"mix":221},{"t":384.334,
"mix":222},{"t":386.143,"mix":223},{"t":387.884,"mix":224},{"t":389.578,"mix":225},{"t":391.423,
"mix":226},{"t":393.193,"mix":227},{"t":395.211,"mix":228},{"t":397.034,"mix":229},{"t":398.828,
"mix":230},{"t":400.537,"mix":231},{"t":402.316,"mix":232},{"t":404.168,"mix":233},{"t":405.989,
"mix":234},{"t":407.834,"mix":235},{"t":409.774,"mix":236},{"t":411.747,"mix":237},{"t":414.067,
"mix":238},{"t":416.017,"mix":239},{"t":418.244,"mix":240},{"t":420.525,"mix":241},{"t":422.921,
"mix":242},{"t":425.806,"mix":243},{"t":428.235,"mix":244},{"t":430.613,"mix":245},{"t":433.344,
"mix":246},{"t":436.315,"mix":247},{"t":438.355,"mix":248},{"t":441.224,"mix":249},{"t":443.555,
"mix":250},{"t":445.427,"mix":251},{"t":447.334,"mix":252},{"t":450.226,"mix":253},{"t":451.962,
"mix":254},{"t":454.167,"mix":255},{"t":456.455,"mix":256},{"t":458.253,"mix":257},{"t":460.543,
"mix":258},{"t":462.852,"mix":259},{"t":464.813,"mix":260},{"t":466.574,"mix":261},{"t":468.158,
"mix":262},{"t":469.849,"mix":263},{"t":471.624,"mix":264},{"t":473.191,"mix":265},{"t":474.955,
"mix":266},{"t":476.597,"mix":267},{"t":478.281,"mix":268},{"t":480.168,"mix":269},{"t":481.962,
"mix":270},{"t":483.802,"mix":271},{"t":485.974,"mix":272},{"t":488.139,"mix":273},{"t":490.045,
"mix":274},{"t":491.684,"mix":275},{"t":493.364,"mix":276},{"t":495.046,"mix":277},{"t":496.696,
"mix":278},{"t":498.261,"mix":279},{"t":499.886,"mix":280},{"t":501.52,"mix":281},{"t":503.165,
"mix":282},{"t":504.751,"mix":283},{"t":506.39,"mix":284},{"t":508.014,"mix":285},{"t":509.627,
"mix":286},{"t":511.294,"mix":287},{"t":512.971,"mix":288},{"t":514.954,"mix":289},{"t":516.884,
"mix":290},{"t":518.568,"mix":291},{"t":520.172,"mix":292},{"t":521.782,"mix":293},{"t":523.429,
"mix":294},{"t":525.165,"mix":295},{"t":526.668,"mix":296},{"t":528.394,"mix":297},{"t":529.854,
"mix":298},{"t":531.455,"mix":299},{"t":533.027,"mix":300},{"t":534.523,"mix":301},{"t":536.126,
"mix":302},{"t":537.649,"mix":303},{"t":539.183,"mix":304},{"t":540.763,"mix":305},{"t":542.249,
"mix":306},{"t":543.786,"mix":307},{"t":545.316,"mix":308},{"t":546.819,"mix":309},{"t":548.388,
"mix":310},{"t":549.822,"mix":311},{"t":551.418,"mix":312},{"t":552.932,"mix":313},{"t":554.442,
"mix":314},{"t":555.902,"mix":315},{"t":557.37,"mix":316},{"t":558.887,"mix":317},{"t":560.513,
"mix":318},{"t":562.131,"mix":319},{"t":563.687,"mix":320},{"t":565.376,"mix":321},{"t":567.091,
"mix":322},{"t":568.635,"mix":323},{"t":570.201,"mix":324},{"t":571.791,"mix":325},{"t":573.279,
"mix":326},{"t":574.783,"mix":327},{"t":576.249,"mix":328},{"t":577.646,"mix":329},{"t":579.228,
"mix":330},{"t":580.69,"mix":331},{"t":582.291,"mix":332},{"t":583.907,"mix":333},{"t":585.456,
"mix":334},{"t":587.306,"mix":335},{"t":589.093,"mix":336},{"t":591.036,"mix":337},{"t":592.879,
"mix":338},{"t":595.387,"mix":339},{"t":597.059,"mix":340},{"t":598.772,"mix":341},{"t":600.454,
"mix":342},{"t":602.151,"mix":343},{"t":603.808,"mix":344},{"t":605.484,"mix":345},{"t":607.113,
"mix":346},{"t":608.789,"mix":347},{"t":610.625,"mix":348},{"t":612.284,"mix":349},{"t":614.142,
"mix":350},{"t":616.145,"mix":351},{"t":617.994,"mix":352},{"t":619.766,"mix":353},{"t":621.748,
"mix":354},{"t":624.091,"mix":355},{"t":626.042,"mix":356},{"t":627.672,"mix":357},{"t":629.503,
"mix":358},{"t":631.545,"mix":359},{"t":633.17,"mix":360},{"t":634.765,"mix":361},{"t":636.328,
"mix":362},{"t":637.989,"mix":363},{"t":639.606,"mix":364},{"t":641.387,"mix":365},{"t":643.08,
"mix":366},{"t":644.735,"mix":367},{"t":646.394,"mix":368},{"t":647.838,"mix":369},{"t":649.489,
"mix":370},{"t":651.201,"mix":371},{"t":652.762,"mix":372},{"t":654.284,"mix":373},{"t":655.738,
"mix":374},{"t":657.359,"mix":375},{"t":658.974,"mix":376},{"t":660.51,"mix":377},{"t":662.176,
"mix":378},{"t":663.663,"mix":379},{"t":665.133,"mix":380},{"t":666.727,"mix":381},{"t":668.318,
"mix":382},{"t":669.738,"mix":383},{"t":671.401,"mix":384},{"t":672.995,"mix":385},{"t":674.512,
"mix":386},{"t":676.261,"mix":387},{"t":677.626,"mix":388},{"t":679.23,"mix":389},{"t":680.618,
"mix":390},{"t":682.372,"mix":391},{"t":684.045,"mix":392},{"t":685.502,"mix":393},{"t":687.251,
"mix":394},{"t":688.871,"mix":395},{"t":690.429,"mix":396},{"t":692.018,"mix":397},{"t":693.611,
"mix":398},{"t":695.173,"mix":399},{"t":696.692,"mix":400},{"t":698.17,"mix":401},{"t":699.593,
"mix":402},{"t":701.223,"mix":403},{"t":702.751,"mix":404},{"t":704.345,"mix":405},{"t":705.822,
"mix":406},{"t":707.254,"mix":407},{"t":708.748,"mix":408},{"t":710.138,"mix":409},{"t":711.653,
"mix":410},{"t":713.137,"mix":411},{"t":714.625,"mix":412},{"t":716.09,"mix":413},{"t":717.554,
"mix":414},{"t":719.124,"mix":415},{"t":720.641,"mix":416},{"t":722.034,"mix":417},{"t":723.572,
"mix":418},{"t":725.24,"mix":419},{"t":726.77,"mix":420},{"t":728.185,"mix":421},{"t":729.684,
"mix":422},{"t":731.324,"mix":423},{"t":732.739,"mix":424},{"t":734.151,"mix":425},{"t":735.623,
"mix":426},{"t":737.086,"mix":427},{"t":738.484,"mix":428},{"t":739.98,"mix":429},{"t":741.475,
"mix":430},{"t":743.148,"mix":431},{"t":744.766,"mix":432},{"t":746.432,"mix":433},{"t":748.079,
"mix":434},{"t":750.141,"mix":435},{"t":751.952,"mix":436},{"t":753.649,"mix":437},{"t":755.373,
"mix":438},{"t":757.242,"mix":439},{"t":759.535,"mix":440},{"t":771.378,"mix":441},{"t":777.654,
"mix":442},{"t":783.174,"mix":443},{"t":788.564,"mix":444},{"t":794.55,"mix":445},{"t":800.11,
"mix":446},{"t":805.556,"mix":447},{"t":811.045,"mix":448},{"t":816.573,"mix":449},{"t":821.971,
"mix":450},{"t":827.685,"mix":451},{"t":833.643,"mix":452},{"t":839.87,"mix":453},{"t":845.697,
"mix":454},{"t":851.247,"mix":455},{"t":856.26,"mix":456},{"t":861.154,"mix":457},{"t":866.259,
"mix":458},{"t":871.097,"mix":459},{"t":876.036,"mix":460},{"t":881.217,"mix":461},{"t":886.964,
"mix":462},{"t":892.547,"mix":463},{"t":898.219,"mix":464},{"t":903.62,"mix":465},{"t":909.117,
"mix":466},{"t":914.634,"mix":467},{"t":920.5,"mix":468},{"t":926.903,"mix":469},{"t":933.891,
"mix":470},{"t":940.11,"mix":471},{"t":945.933,"mix":472},{"t":951.43,"mix":473},{"t":956.633,
"mix":474},{"t":961.919,"mix":475},{"t":967.322,"mix":476},{"t":972.46,"mix":477},{"t":977.658,
"mix":478},{"t":982.846,"mix":479},{"t":988.903,"mix":480},{"t":995.146,"mix":481},{"t":1000.975,
"mix":482},{"t":1006.563,"mix":483},{"t":1012.226,"mix":484},{"t":1017.839,"mix":485},{"t":1023.405,
"mix":486},{"t":1029.452,"mix":487},{"t":1035.71,"mix":488},{"t":1041.512,"mix":489},{"t":1047.9,
"mix":490},{"t":1053.694,"mix":491},{"t":1059.722,"mix":492},{"t":1064.713,"mix":493},{"t":1069.817,
"mix":494},{"t":1075.051,"mix":495},{"t":1079.646,"mix":496},{"t":1084.589,"mix":497},{"t":1089.714,
"mix":498},{"t":1094.978,"mix":499},{"t":1100.65,"mix":500},{"t":1106.319,"mix":501},{"t":1111.962,
"mix":502},{"t":1117.406,"mix":503},{"t":1123.175,"mix":504},{"t":1129.024,"mix":505},{"t":1134.634,
"mix":506},{"t":1140.147,"mix":507},{"t":1146.037,"mix":508},{"t":1151.441,"mix":509},{"t":1156.87,
"mix":510},{"t":1162.672,"mix":511},{"t":1168.853,"mix":512},{"t":1174.544,"mix":513},{"t":1179.936,
"mix":514},{"t":1184.985,"mix":515},{"t":1189.938,"mix":516},{"t":1194.98,"mix":517},{"t":1200.049,
"mix":518},{"t":1204.881,"mix":519},{"t":1209.903,"mix":520},{"t":1215.268,"mix":521},{"t":1220.359,
"mix":522},{"t":1225.357,"mix":523},{"t":1230.868,"mix":524},{"t":1236.568,"mix":525},{"t":1242.344,
"mix":526},{"t":1248.255,"mix":527},{"t":1254.852,"mix":528},{"t":1260.142,"mix":529},{"t":1265.588,
"mix":530},{"t":1270.635,"mix":531},{"t":1276.425,"mix":532},{"t":1281.65,"mix":533},{"t":1287.648,
"mix":534},{"t":1293.827,"mix":535},{"t":1300.055,"mix":536},{"t":1305.985,"mix":537},{"t":1311.49,
"mix":538},{"t":1317.483,"mix":539},{"t":1323.204,"mix":540},{"t":1328.86,"mix":541},{"t":1335.379,
"mix":542},{"t":1341.437,"mix":543},{"t":1347.527,"mix":544},{"t":1353.651,"mix":545},{"t":1360.621,
"mix":546},{"t":1366.131,"mix":547},{"t":1371.923,"mix":548},{"t":1378.082,"mix":549},{"t":1385.304,
"mix":550},{"t":1394.745,"mix":551},{"t":1400.971,"mix":552},{"t":1410.913,"mix":553},{"t":1417.284,
"mix":554},{"t":1423.577,"mix":555},{"t":1430.273,"mix":556},{"t":1438.471,"mix":557},{"t":1446.732,
"mix":558},{"t":1455.117,"mix":559},{"t":1456.86,"mix":560},{"t":1458.003,"mix":561},{"t":1458.979,
"mix":562},{"t":1460.206,"mix":563},{"t":1461.342,"mix":564},{"t":1462.271,"mix":565},{"t":1463.263,
"mix":566},{"t":1464.236,"mix":567},{"t":1465.22,"mix":568},{"t":1466.191,"mix":569},{"t":1467.212,
"mix":570},{"t":1468.217,"mix":571},{"t":1469.204,"mix":572},{"t":1470.265,"mix":573},{"t":1471.238,
"mix":574},{"t":1472.228,"mix":575},{"t":1473.176,"mix":576},{"t":1474.078,"mix":577},{"t":1475.109,
"mix":578},{"t":1476.12,"mix":579},{"t":1477.14,"mix":580},{"t":1478.11,"mix":581},{"t":1479.137,
"mix":582},{"t":1480.185,"mix":583},{"t":1481.183,"mix":584},{"t":1482.126,"mix":585},{"t":1483.101,
"mix":586},{"t":1484.064,"mix":587},{"t":1484.974,"mix":588},{"t":1485.855,"mix":589},{"t":1486.764,
"mix":590},{"t":1487.708,"mix":591},{"t":1488.607,"mix":592},{"t":1489.453,"mix":593},{"t":1490.417,
"mix":594},{"t":1491.485,"mix":595},{"t":1492.476,"mix":596},{"t":1493.686,"mix":597},{"t":1494.748,
"mix":598},{"t":1495.716,"mix":599},{"t":1496.684,"mix":600},{"t":1497.662,"mix":601},{"t":1498.594,
"mix":602},{"t":1499.646,"mix":603},{"t":1500.608,"mix":604},{"t":1501.617,"mix":605},{"t":1502.58,
"mix":606},{"t":1503.619,"mix":607},{"t":1504.584,"mix":608},{"t":1505.521,"mix":609},{"t":1506.552,
"mix":610},{"t":1507.552,"mix":611},{"t":1508.642,"mix":612},{"t":1509.666,"mix":613},{"t":1510.702,
"mix":614},{"t":1511.641,"mix":615},{"t":1512.624,"mix":616},{"t":1513.664,"mix":617},{"t":1514.7,
"mix":618},{"t":1515.634,"mix":619},{"t":1516.64,"mix":620},{"t":1517.64,"mix":621},{"t":1518.599,
"mix":622},{"t":1519.581,"mix":623},{"t":1520.482,"mix":624},{"t":1521.481,"mix":625},{"t":1522.468,
"mix":626},{"t":1523.438,"mix":627},{"t":1524.361,"mix":628},{"t":1525.312,"mix":629},{"t":1526.249,
"mix":630},{"t":1527.145,"mix":631},{"t":1528.1,"mix":632},{"t":1529.107,"mix":633},{"t":1530.148,
"mix":634},{"t":1531.113,"mix":635},{"t":1532.055,"mix":636},{"t":1533.064,"mix":637},{"t":1534.025,
"mix":638},{"t":1535.219,"mix":639},{"t":1536.23,"mix":640},{"t":1537.245,"mix":641},{"t":1538.259,
"mix":642},{"t":1539.174,"mix":643},{"t":1540.17,"mix":644},{"t":1541.202,"mix":645},{"t":1542.149,
"mix":646},{"t":1543.181,"mix":647},{"t":1544.125,"mix":648},{"t":1545.181,"mix":649},{"t":1546.138,
"mix":650},{"t":1547.286,"mix":651},{"t":1548.42,"mix":652},{"t":1549.354,"mix":653},{"t":1550.349,
"mix":654},{"t":1551.238,"mix":655},{"t":1552.431,"mix":656},{"t":1553.309,"mix":657},{"t":1554.389,
"mix":658},{"t":1555.317,"mix":659},{"t":1556.465,"mix":660},{"t":1557.321,"mix":661},{"t":1558.329,
"mix":662},{"t":1559.233,"mix":663},{"t":1560.183,"mix":664},{"t":1561.244,"mix":665},{"t":1562.237,
"mix":666},{"t":1563.247,"mix":667},{"t":1564.2,"mix":668},{"t":1565.237,"mix":669},{"t":1566.1,
"mix":670},{"t":1567.069,"mix":671},{"t":1568.033,"mix":672},{"t":1569.055,"mix":673},{"t":1569.94,
"mix":674},{"t":1570.884,"mix":675},{"t":1571.947,"mix":676},{"t":1573.005,"mix":677},{"t":1573.98,
"mix":678},{"t":1574.997,"mix":679},{"t":1575.961,"mix":680},{"t":1576.888,"mix":681},{"t":1577.923,
"mix":682},{"t":1578.744,"mix":683},{"t":1579.686,"mix":684},{"t":1580.622,"mix":685},{"t":1581.567,
"mix":686},{"t":1582.505,"mix":687},{"t":1583.521,"mix":688},{"t":1584.411,"mix":689},{"t":1585.362,
"mix":690},{"t":1586.272,"mix":691},{"t":1587.236,"mix":692},{"t":1588.181,"mix":693},{"t":1589.084,
"mix":694},{"t":1589.957,"mix":695},{"t":1590.87,"mix":696},{"t":1591.735,"mix":697},{"t":1592.632,
"mix":698},{"t":1593.587,"mix":699},{"t":1594.493,"mix":700},{"t":1595.442,"mix":701},{"t":1596.42,
"mix":702},{"t":1597.436,"mix":703},{"t":1598.397,"mix":704},{"t":1599.331,"mix":705},{"t":1600.378,
"mix":706},{"t":1601.343,"mix":707},{"t":1602.272,"mix":708},{"t":1603.227,"mix":709},{"t":1604.227,
"mix":710},{"t":1605.144,"mix":711},{"t":1606.154,"mix":712},{"t":1607.162,"mix":713},{"t":1608.174,
"mix":714},{"t":1609.136,"mix":715},{"t":1610.132,"mix":716},{"t":1611.15,"mix":717},{"t":1612.141,
"mix":718},{"t":1613.097,"mix":719},{"t":1614.111,"mix":720},{"t":1615.108,"mix":721},{"t":1616.119,
"mix":722},{"t":1617.111,"mix":723},{"t":1618.057,"mix":724},{"t":1619.023,"mix":725},{"t":1619.99,
"mix":726},{"t":1621.052,"mix":727},{"t":1621.994,"mix":728},{"t":1622.934,"mix":729},{"t":1623.847,
"mix":730},{"t":1624.882,"mix":731},{"t":1625.842,"mix":732},{"t":1627.014,"mix":733},{"t":1628.151,
"mix":734},{"t":1629.379,"mix":735},{"t":1630.667,"mix":736},{"t":1631.918,"mix":737},{"t":1633.045,
"mix":738},{"t":1634.32,"mix":739},{"t":1635.858,"mix":740},{"t":1637.102,"mix":741},{"t":1638.488,
"mix":742},{"t":1639.851,"mix":743},{"t":1641.132,"mix":744},{"t":1642.475,"mix":745},{"t":1643.741,
"mix":746},{"t":1645.153,"mix":747},{"t":1646.675,"mix":748},{"t":1648.033,"mix":749},{"t":1649.447,
"mix":750},{"t":1650.828,"mix":751},{"t":1652.396,"mix":752},{"t":1653.676,"mix":753},{"t":1655.223,
"mix":754},{"t":1656.585,"mix":755},{"t":1658.283,"mix":756},{"t":1660.04,"mix":757},{"t":1661.425,
"mix":758},{"t":1662.515,"mix":759},{"t":1663.555,"mix":760},{"t":1664.589,"mix":761},{"t":1665.648,
"mix":762},{"t":1666.619,"mix":763},{"t":1667.628,"mix":764},{"t":1668.585,"mix":765},{"t":1669.579,
"mix":766},{"t":1670.561,"mix":767},{"t":1671.616,"mix":768},{"t":1672.681,"mix":769},{"t":1673.71,
"mix":770},{"t":1674.719,"mix":771},{"t":1675.74,"mix":772},{"t":1676.72,"mix":773},{"t":1677.656,
"mix":774},{"t":1678.685,"mix":775},{"t":1679.674,"mix":776},{"t":1680.591,"mix":777},{"t":1681.572,
"mix":778},{"t":1682.523,"mix":779},{"t":1683.447,"mix":780},{"t":1684.436,"mix":781},{"t":1685.373,
"mix":782},{"t":1686.249,"mix":783},{"t":1687.283,"mix":784},{"t":1688.214,"mix":785},{"t":1689.276,
"mix":786},{"t":1690.204,"mix":787},{"t":1691.151,"mix":788},{"t":1692.077,"mix":789},{"t":1693.303,
"mix":790},{"t":1694.362,"mix":791},{"t":1695.372,"mix":792},{"t":1696.361,"mix":793},{"t":1697.323,
"mix":794},{"t":1698.359,"mix":795},{"t":1699.323,"mix":796},{"t":1700.265,"mix":797},{"t":1701.228,
"mix":798},{"t":1702.203,"mix":799},{"t":1703.194,"mix":800},{"t":1704.181,"mix":801},{"t":1705.132,
"mix":802},{"t":1706.093,"mix":803},{"t":1707.047,"mix":804},{"t":1707.987,"mix":805},{"t":1709.081,
"mix":806},{"t":1710.056,"mix":807},{"t":1711.079,"mix":808},{"t":1712.094,"mix":809},{"t":1713.073,
"mix":810},{"t":1714.073,"mix":811},{"t":1715.119,"mix":812},{"t":1716.094,"mix":813},{"t":1717.119,
"mix":814},{"t":1718.07,"mix":815},{"t":1719.106,"mix":816},{"t":1720.078,"mix":817},{"t":1721.043,
"mix":818},{"t":1722.004,"mix":819},{"t":1722.942,"mix":820},{"t":1723.946,"mix":821},{"t":1724.888,
"mix":822},{"t":1725.844,"mix":823},{"t":1726.78,"mix":824},{"t":1727.745,"mix":825},{"t":1728.763,
"mix":826},{"t":1729.694,"mix":827},{"t":1730.662,"mix":828},{"t":1731.639,"mix":829},{"t":1732.602,
"mix":830},{"t":1733.588,"mix":831},{"t":1734.515,"mix":832},{"t":1735.445,"mix":833},{"t":1736.455,
"mix":834},{"t":1737.414,"mix":835},{"t":1738.402,"mix":836},{"t":1739.338,"mix":837},{"t":1740.425,
"mix":838},{"t":1741.377,"mix":839},{"t":1742.302,"mix":840},{"t":1743.323,"mix":841},{"t":1744.243,
"mix":842},{"t":1745.19,"mix":843},{"t":1746.12,"mix":844},{"t":1747.098,"mix":845},{"t":1748.08,
"mix":846},{"t":1749.02,"mix":847},{"t":1749.962,"mix":848},{"t":1750.979,"mix":849},{"t":1751.931,
"mix":850},{"t":1752.869,"mix":851},{"t":1753.843,"mix":852},{"t":1754.797,"mix":853},{"t":1755.845,
"mix":854},{"t":1756.796,"mix":855},{"t":1757.751,"mix":856},{"t":1758.687,"mix":857},{"t":1759.695,
"mix":858},{"t":1760.609,"mix":859},{"t":1761.547,"mix":860},{"t":1762.389,"mix":861},{"t":1763.281,
"mix":862},{"t":1764.155,"mix":863},{"t":1765.151,"mix":864},{"t":1766.079,"mix":865},{"t":1767.029,
"mix":866},{"t":1767.921,"mix":867},{"t":1768.837,"mix":868},{"t":1769.721,"mix":869},{"t":1770.696,
"mix":870},{"t":1771.791,"mix":871},{"t":1772.739,"mix":872},{"t":1773.886,"mix":873},{"t":1774.849,
"mix":874},{"t":1775.955,"mix":875},{"t":1777.055,"mix":876},{"t":1778.008,"mix":877},{"t":1779.137,
"mix":878},{"t":1780.192,"mix":879},{"t":1781.204,"mix":880},{"t":1782.105,"mix":881},{"t":1783.054,
"mix":882},{"t":1784.063,"mix":883},{"t":1785.073,"mix":884},{"t":1786.047,"mix":885},{"t":1787.005,
"mix":886},{"t":1787.99,"mix":887},{"t":1788.893,"mix":888},{"t":1789.919,"mix":889},{"t":1790.811,
"mix":890},{"t":1791.75,"mix":891},{"t":1792.73,"mix":892},{"t":1793.72,"mix":893},{"t":1794.634,
"mix":894},{"t":1795.596,"mix":895},{"t":1796.664,"mix":896},{"t":1797.606,"mix":897},{"t":1798.538,
"mix":898},{"t":1799.462,"mix":899},{"t":1800.466,"mix":900},{"t":1801.41,"mix":901},{"t":1802.434,
"mix":902},{"t":1803.356,"mix":903},{"t":1804.344,"mix":904},{"t":1805.212,"mix":905},{"t":1806.208,
"mix":906},{"t":1807.321,"mix":907},{"t":1808.333,"mix":908},{"t":1809.359,"mix":909},{"t":1810.417,
"mix":910},{"t":1811.569,"mix":911},{"t":1812.623,"mix":912},{"t":1813.637,"mix":913},{"t":1814.654,
"mix":914},{"t":1815.757,"mix":915},{"t":1824.517,"mix":916},{"t":1826.813,"mix":917},{"t":1828.365,
"mix":918},{"t":1830.169,"mix":919},{"t":1831.677,"mix":920},{"t":1833.367,"mix":921},{"t":1834.977,
"mix":922},{"t":1836.817,"mix":923},{"t":1838.53,"mix":924},{"t":1840.325,"mix":925},{"t":1842.117,
"mix":926},{"t":1843.823,"mix":927},{"t":1845.62,"mix":928},{"t":1847.31,"mix":929},{"t":1849.117,
"mix":930},{"t":1850.987,"mix":931},{"t":1852.738,"mix":932},{"t":1854.264,"mix":933},{"t":1855.777,
"mix":934},{"t":1857.308,"mix":935},{"t":1858.848,"mix":936},{"t":1860.374,"mix":937},{"t":1861.855,
"mix":938},{"t":1863.347,"mix":939},{"t":1864.969,"mix":940},{"t":1866.526,"mix":941},{"t":1868.017,
"mix":942},{"t":1869.521,"mix":943},{"t":1871.088,"mix":944},{"t":1872.552,"mix":945},{"t":1874.114,
"mix":946},{"t":1875.604,"mix":947},{"t":1877.33,"mix":948},{"t":1879.123,"mix":949},{"t":1880.785,
"mix":950},{"t":1882.711,"mix":951},{"t":1884.568,"mix":952},{"t":1886.359,"mix":953},{"t":1888.206,
"mix":954},{"t":1889.904,"mix":955},{"t":1892.113,"mix":956},{"t":1894.019,"mix":957},{"t":1895.679,
"mix":958},{"t":1897.32,"mix":959},{"t":1898.778,"mix":960},{"t":1900.333,"mix":961},{"t":1901.968,
"mix":962},{"t":1903.616,"mix":963},{"t":1905.49,"mix":964},{"t":1907.184,"mix":965},{"t":1908.937,
"mix":966},{"t":1910.514,"mix":967},{"t":1912.137,"mix":968},{"t":1913.727,"mix":969},{"t":1915.289,
"mix":970},{"t":1916.959,"mix":971},{"t":1919.281,"mix":972},{"t":1921.354,"mix":973},{"t":1923.358,
"mix":974},{"t":1925.424,"mix":975},{"t":1927.423,"mix":976},{"t":1929.527,"mix":977},{"t":1931.683,
"mix":978},{"t":1933.744,"mix":979},{"t":1935.734,"mix":980},{"t":1937.801,"mix":981},{"t":1939.841,
"mix":982},{"t":1941.673,"mix":983},{"t":1943.545,"mix":984},{"t":1945.48,"mix":985},{"t":1947.345,
"mix":986},{"t":1949.413,"mix":987},{"t":1951.434,"mix":988},{"t":1953.543,"mix":989},{"t":1955.411,
"mix":990},{"t":1957.402,"mix":991},{"t":1959.256,"mix":992},{"t":1961.268,"mix":993},{"t":1963.177,
"mix":994},{"t":1965.304,"mix":995},{"t":1968.157,"mix":996},{"t":1970.358,"mix":997},{"t":1972.289,
"mix":998},{"t":1973.949,"mix":999},{"t":1975.835,"mix":1000},{"t":1978.058,"mix":1001},{"t":1980.096,
"mix":1002},{"t":1982.73,"mix":1003},{"t":1985.064,"mix":1004},{"t":1987.222,"mix":1005},
{"t":1989.297,"mix":1006},{"t":1991.434,"mix":1007},{"t":1993.534,"mix":1008},{"t":1995.915,
"mix":1009},{"t":1997.978,"mix":1010},{"t":2000.101,"mix":1011},{"t":2002.641,"mix":1012},
{"t":2007.32,"mix":1013},{"t":2011.653,"mix":1014},{"t":2016.283,"mix":1015},{"t":2020.655,
"mix":1016},{"t":2024.791,"mix":1017},{"t":2029.267,"mix":1018},{"t":2034.071,"mix":1019},
{"t":2039.518,"mix":1020},{"t":2044.284,"mix":1021},{"t":2048.921,"mix":1022},{"t":2053.303,
"mix":1023},{"t":2057.786,"mix":1024},{"t":2062.056,"mix":1025},{"t":2066.568,"mix":1026},
{"t":2071.011,"mix":1027},{"t":2075.464,"mix":1028},{"t":2080.144,"mix":1029},{"t":2084.203,
"mix":1030},{"t":2088.21,"mix":1031},{"t":2092.04,"mix":1032},{"t":2095.84,"mix":1033},{"t":2099.844,
"mix":1034},{"t":2104.096,"mix":1035},{"t":2108.614,"mix":1036},{"t":2112.599,"mix":1037},
{"t":2116.125,"mix":1038},{"t":2119.947,"mix":1039},{"t":2123.569,"mix":1040},{"t":2127.147,
"mix":1041},{"t":2131.083,"mix":1042},{"t":2135.56,"mix":1043},{"t":2145.852,"mix":1044},
{"t":2147.401,"mix":1045},{"t":2148.639,"mix":1046},{"t":2149.825,"mix":1047},{"t":2151.323,
"mix":1048},{"t":2152.553,"mix":1049},{"t":2153.719,"mix":1050},{"t":2154.994,"mix":1051},
{"t":2156.638,"mix":1052},{"t":2158.22,"mix":1053},{"t":2159.395,"mix":1054},{"t":2160.561,
"mix":1055},{"t":2161.715,"mix":1056},{"t":2162.869,"mix":1057},{"t":2164.201,"mix":1058},
{"t":2165.366,"mix":1059},{"t":2166.672,"mix":1060},{"t":2168.122,"mix":1061},{"t":2169.479,
"mix":1062},{"t":2170.835,"mix":1063},{"t":2172.088,"mix":1064},{"t":2173.449,"mix":1065},
{"t":2174.689,"mix":1066},{"t":2175.975,"mix":1067},{"t":2177.289,"mix":1068},{"t":2178.572,
"mix":1069},{"t":2180.004,"mix":1070},{"t":2181.35,"mix":1071},{"t":2182.684,"mix":1072},
{"t":2184.065,"mix":1073},{"t":2185.352,"mix":1074},{"t":2186.584,"mix":1075},{"t":2187.926,
"mix":1076},{"t":2189.395,"mix":1077},{"t":2190.734,"mix":1078},{"t":2192.144,"mix":1079},
{"t":2193.498,"mix":1080},{"t":2194.825,"mix":1081},{"t":2196.213,"mix":1082},{"t":2197.502,
"mix":1083},{"t":2198.839,"mix":1084},{"t":2200.338,"mix":1085},{"t":2201.676,"mix":1086},
{"t":2203.102,"mix":1087},{"t":2204.501,"mix":1088},{"t":2205.716,"mix":1089},{"t":2207.054,
"mix":1090},{"t":2208.538,"mix":1091},{"t":2209.685,"mix":1092},{"t":2211.009,"mix":1093},
{"t":2212.278,"mix":1094},{"t":2213.558,"mix":1095},{"t":2214.774,"mix":1096},{"t":2215.842,
"mix":1097},{"t":2217.019,"mix":1098},{"t":2218.245,"mix":1099},{"t":2219.559,"mix":1100},
{"t":2220.9,"mix":1101},{"t":2222.163,"mix":1102},{"t":2223.504,"mix":1103},{"t":2224.818,
"mix":1104},{"t":2226.209,"mix":1105},{"t":2227.519,"mix":1106},{"t":2228.829,"mix":1107},
{"t":2230.776,"mix":1108},{"t":2232.317,"mix":1109},{"t":2233.88,"mix":1110},{"t":2235.482,
"mix":1111},{"t":2237.097,"mix":1112},{"t":2238.645,"mix":1113},{"t":2240.143,"mix":1114},
{"t":2241.746,"mix":1115},{"t":2243.317,"mix":1116},{"t":2244.866,"mix":1117},{"t":2246.291,
"mix":1118},{"t":2247.814,"mix":1119},{"t":2249.286,"mix":1120},{"t":2250.766,"mix":1121},
{"t":2252.175,"mix":1122},{"t":2253.611,"mix":1123},{"t":2255.102,"mix":1124},{"t":2256.75,
"mix":1125},{"t":2258.184,"mix":1126},{"t":2259.549,"mix":1127},{"t":2260.87,"mix":1128},
{"t":2262.35,"mix":1129},{"t":2263.682,"mix":1130},{"t":2265.192,"mix":1131},{"t":2266.828,
"mix":1132},{"t":2268.353,"mix":1133},{"t":2269.683,"mix":1134},{"t":2271.201,"mix":1135},
{"t":2272.512,"mix":1136},{"t":2273.915,"mix":1137},{"t":2275.299,"mix":1138},{"t":2276.658,
"mix":1139},{"t":2278.166,"mix":1140},{"t":2279.573,"mix":1141},{"t":2280.95,"mix":1142},
{"t":2282.257,"mix":1143},{"t":2283.583,"mix":1144},{"t":2284.768,"mix":1145},{"t":2286.124,
"mix":1146},{"t":2287.382,"mix":1147},{"t":2288.88,"mix":1148},{"t":2290.359,"mix":1149},
{"t":2291.808,"mix":1150},{"t":2293.109,"mix":1151},{"t":2294.542,"mix":1152},{"t":2296.151,
"mix":1153},{"t":2297.519,"mix":1154},{"t":2298.924,"mix":1155},{"t":2300.4,"mix":1156},{"t":2301.904,
"mix":1157},{"t":2303.309,"mix":1158},{"t":2304.567,"mix":1159},{"t":2305.812,"mix":1160},
{"t":2307.308,"mix":1161},{"t":2308.554,"mix":1162},{"t":2309.932,"mix":1163},{"t":2311.15,
"mix":1164},{"t":2312.635,"mix":1165},{"t":2314.121,"mix":1166},{"t":2315.541,"mix":1167},
{"t":2317.438,"mix":1168},{"t":2318.582,"mix":1169},{"t":2319.609,"mix":1170},{"t":2320.763,
"mix":1171},{"t":2321.777,"mix":1172},{"t":2322.853,"mix":1173},{"t":2323.792,"mix":1174},
{"t":2324.888,"mix":1175},{"t":2325.883,"mix":1176},{"t":2326.967,"mix":1177},{"t":2327.969,
"mix":1178},{"t":2328.978,"mix":1179},{"t":2329.994,"mix":1180},{"t":2330.988,"mix":1181},
{"t":2331.916,"mix":1182},{"t":2332.921,"mix":1183},{"t":2333.78,"mix":1184},{"t":2334.785,
"mix":1185},{"t":2335.717,"mix":1186},{"t":2336.609,"mix":1187},{"t":2337.621,"mix":1188},
{"t":2338.531,"mix":1189},{"t":2339.529,"mix":1190},{"t":2340.457,"mix":1191},{"t":2341.32,
"mix":1192},{"t":2342.256,"mix":1193},{"t":2343.105,"mix":1194},{"t":2344.061,"mix":1195},
{"t":2344.876,"mix":1196},{"t":2345.868,"mix":1197},{"t":2346.779,"mix":1198},{"t":2347.685,
"mix":1199},{"t":2348.538,"mix":1200},{"t":2349.487,"mix":1201},{"t":2350.329,"mix":1202},
{"t":2351.207,"mix":1203},{"t":2352.041,"mix":1204},{"t":2352.899,"mix":1205},{"t":2353.775,
"mix":1206},{"t":2354.653,"mix":1207},{"t":2355.606,"mix":1208},{"t":2356.532,"mix":1209},
{"t":2357.374,"mix":1210},{"t":2358.281,"mix":1211},{"t":2359.112,"mix":1212},{"t":2360.044,
"mix":1213},{"t":2360.97,"mix":1214},{"t":2361.859,"mix":1215},{"t":2362.87,"mix":1216},{"t":2363.772,
"mix":1217},{"t":2364.743,"mix":1218},{"t":2365.673,"mix":1219},{"t":2366.52,"mix":1220},
{"t":2367.49,"mix":1221},{"t":2368.456,"mix":1222},{"t":2369.36,"mix":1223},{"t":2370.29,
"mix":1224},{"t":2371.33,"mix":1225},{"t":2372.389,"mix":1226},{"t":2379.991,"mix":1227}]
'''  # Use the full JSON
timestamps = json.loads(timestamps_json)

# Sample JSON timestamps for the first recording (assumed already loaded)
offset = 3.36
adjusted_timestamps = adjust_timestamps(warping_path, timestamps, sr1, offset)

# Adjust so that the first timestamp is zero
initial_offset = -adjusted_timestamps[0]['t']
new_adjusted_timestamps = [{"t": item["t"] + initial_offset, "mix": item["mix"]} for item in adjusted_timestamps]

# Print the adjusted timestamps and initial offset
print(json.dumps(new_adjusted_timestamps, indent=4))
print(f"Offset: {initial_offset}")

# Cleanup downloaded and converted files
os.remove(audio_path1)
os.remove(audio_path2)
os.remove(ogg_path1)
os.remove(ogg_path2)

URLError: <urlopen error [Errno 113] No route to host>